# Mammography Model Inference

**Load trained ResNet-50 model and make predictions**

## Features:
- Load trained checkpoint from `Simple_Training_VinDr.ipynb`
- Evaluate on test set
- Make predictions on new images
- Breast-level aggregation using Noisy-OR
- Visualize predictions with confidence scores

---

## Configuration

In [ ]:
# ============================================================================
# INFERENCE CONFIGURATION
# ============================================================================

# Paths
BASE_DIR = '/content/drive/MyDrive/vindr-mammo'
PREPROCESSED_DIR = f'{BASE_DIR}/preprocessed_png_512'
TEST_METADATA_CSV = f'{PREPROCESSED_DIR}/test.csv'  # Test metadata
CHECKPOINT_PATH = '/content/drive/MyDrive/training_output/best_model.pt'  # Trained model

# Model settings
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64  # Larger batch for inference (no gradient computation)
DEVICE = 'cuda'  # 'cuda' or 'cpu'

print("✅ Configuration loaded")

## Step 1: Setup Environment

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, classification_report
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 3: Define Model Architecture

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

class ResNet50Classifier(nn.Module):
    """ResNet-50 binary classifier."""
    
    def __init__(self, unfreeze_fraction=1.0, dropout_rate=0.0):
        super().__init__()
        self.unfreeze_fraction = unfreeze_fraction
        self.dropout_rate = dropout_rate
        
        # Load pretrained ResNet-50
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Classification head
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(2048, 1)
    
    def forward(self, x):
        features = self.features(x)
        features = self.avgpool(features)
        features = torch.flatten(features, 1)
        features = self.dropout(features)
        logits = self.classifier(features)
        probs = torch.sigmoid(logits).squeeze(1)
        return probs

print("✅ Model architecture defined")

## Step 4: Load Trained Model

In [ ]:
# Load checkpoint
print(f"📦 Loading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

# Get hyperparameters from checkpoint
hyperparams = checkpoint.get('hyperparameters', {})
unfreeze_fraction = hyperparams.get('unfreeze_fraction', 1.0)
dropout_rate = hyperparams.get('dropout_rate', 0.0)

# Create model with same architecture
model = ResNet50Classifier(
    unfreeze_fraction=unfreeze_fraction,
    dropout_rate=dropout_rate
).to(DEVICE)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()  # Set to evaluation mode

print("\n✅ Model loaded successfully!")
print(f"\n📊 Training Metrics (from checkpoint):")
if 'metrics' in checkpoint:
    for key, value in checkpoint['metrics'].items():
        print(f"   {key}: {value:.4f}")

print(f"\n⚙️  Hyperparameters:")
for key, value in hyperparams.items():
    print(f"   {key}: {value}")

print(f"\n🎯 Model is ready for inference!")

## Step 5: Load Test Dataset

In [ ]:
# Load test metadata
print("📊 Loading test dataset...")
test_df = pd.read_csv(TEST_METADATA_CSV)

# Prepare metadata
def prepare_metadata(df):
    """Ensure metadata has all required columns."""
    df = df.copy()
    
    if 'image_id' not in df.columns:
        df['image_id'] = df['png_path'].apply(lambda x: Path(x).stem)
    
    if 'patient_id' not in df.columns and 'study_id' in df.columns:
        df['patient_id'] = df['study_id']
    
    if 'breast_id' not in df.columns:
        if 'laterality' in df.columns:
            df['breast_id'] = df['patient_id'] + '_' + df['laterality']
        else:
            df['breast_id'] = df['patient_id'] + '_Unknown'
    
    if 'png_path' in df.columns and 'image_path' not in df.columns:
        df['image_path'] = df['png_path']
    
    return df

test_df = prepare_metadata(test_df)

print(f"\n✅ Test dataset loaded:")
print(f"   Images: {len(test_df)}")
print(f"   Patients: {test_df['patient_id'].nunique()}")
print(f"   Breasts: {test_df['breast_id'].nunique()}")
print(f"   Malignant: {(test_df['label'] == 1).sum()} ({(test_df['label'] == 1).sum()/len(test_df)*100:.1f}%)")
print(f"   Benign:    {(test_df['label'] == 0).sum()} ({(test_df['label'] == 0).sum()/len(test_df)*100:.1f}%)")

## Step 6: Create Dataset and DataLoader

In [ ]:
class MammogramDataset(Dataset):
    """Dataset for mammogram images."""
    
    def __init__(self, metadata, image_dir, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        
        # Load image
        image_path = os.path.join(self.image_dir, row['image_path'])
        image = Image.open(image_path).convert('L')
        
        # Convert to tensor
        image = torch.from_numpy(np.array(image)).float() / 255.0
        image = image.unsqueeze(0)
        
        # Apply transform
        if self.transform is not None:
            image = self.transform(image)
        
        # Convert to 3-channel for ResNet
        image = image.repeat(3, 1, 1)
        
        label = int(row['label'])
        image_id = row['image_id']
        
        return image, label, image_id

# Create dataset
transform = transforms.Resize(IMAGE_SIZE)
test_dataset = MammogramDataset(
    metadata=test_df,
    image_dir=PREPROCESSED_DIR,
    transform=transform
)

# Create dataloader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoader created: {len(test_loader)} batches")

## Step 7: Run Inference on Test Set

In [ ]:
print("🔄 Running inference on test set...\n")

# Collect predictions
image_predictions = {}
image_labels = {}

with torch.no_grad():
    for images, labels, image_ids in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        predictions = model(images).cpu().numpy()
        
        for img_id, pred, label in zip(image_ids, predictions, labels.numpy()):
            image_predictions[img_id] = float(pred)
            image_labels[img_id] = int(label)

print(f"\n✅ Inference complete: {len(image_predictions)} images processed")

## Step 8: Image-Level Metrics

In [ ]:
# Convert to arrays
img_preds = np.array([image_predictions[img_id] for img_id in test_df['image_id']])
img_labels = np.array([image_labels[img_id] for img_id in test_df['image_id']])

# Compute image-level metrics
img_auroc = roc_auc_score(img_labels, img_preds)
img_pr_auc = average_precision_score(img_labels, img_preds)
img_brier = brier_score_loss(img_labels, img_preds)

print("="*70)
print("IMAGE-LEVEL METRICS")
print("="*70)
print(f"AUROC:  {img_auroc:.4f}")
print(f"PR-AUC: {img_pr_auc:.4f}")
print(f"Brier:  {img_brier:.4f}")
print("="*70)

## Step 9: Breast-Level Aggregation (Noisy-OR)

In [ ]:
def aggregate_to_breast_level(image_predictions, metadata):
    """Aggregate image-level predictions to breast-level using Noisy-OR."""
    breast_preds = {}
    breast_labels = {}
    
    for idx, row in metadata.iterrows():
        breast_id = row['breast_id']
        image_id = row['image_id']
        
        if image_id in image_predictions:
            pred = image_predictions[image_id]
            
            if breast_id not in breast_preds:
                breast_preds[breast_id] = []
                breast_labels[breast_id] = int(row['label'])
            
            breast_preds[breast_id].append(pred)
    
    # Apply Noisy-OR: 1 - prod(1 - p_i)
    final_preds = []
    final_labels = []
    
    for breast_id in breast_preds:
        preds = breast_preds[breast_id]
        noisy_or = 1.0 - np.prod([1.0 - p for p in preds])
        final_preds.append(noisy_or)
        final_labels.append(breast_labels[breast_id])
    
    return np.array(final_preds), np.array(final_labels), breast_preds

# Aggregate to breast level
breast_preds, breast_labels, breast_preds_dict = aggregate_to_breast_level(
    image_predictions, test_df
)

print(f"✅ Aggregated to breast level: {len(breast_preds)} breasts")

## Step 10: Breast-Level Metrics

In [ ]:
# Compute breast-level metrics
breast_auroc = roc_auc_score(breast_labels, breast_preds)
breast_pr_auc = average_precision_score(breast_labels, breast_preds)
breast_brier = brier_score_loss(breast_labels, breast_preds)

print("="*70)
print("BREAST-LEVEL METRICS (Noisy-OR Aggregation)")
print("="*70)
print(f"AUROC:  {breast_auroc:.4f}")
print(f"PR-AUC: {breast_pr_auc:.4f}")
print(f"Brier:  {breast_brier:.4f}")
print(f"\nBreasts evaluated: {len(breast_labels)}")
print(f"  Malignant: {(breast_labels == 1).sum()}")
print(f"  Benign:    {(breast_labels == 0).sum()}")
print("="*70)

## Step 11: Classification Report (with Threshold)

In [ ]:
# Use 0.5 threshold for binary classification
threshold = 0.5
breast_preds_binary = (breast_preds >= threshold).astype(int)

print(f"\n📊 Classification Report (Threshold: {threshold})\n")
print(classification_report(
    breast_labels,
    breast_preds_binary,
    target_names=['Benign', 'Malignant'],
    digits=4
))

# Confusion matrix
cm = confusion_matrix(breast_labels, breast_preds_binary)
print("\nConfusion Matrix:")
print("                Predicted")
print("                Benign  Malignant")
print(f"Actual Benign     {cm[0,0]:4d}    {cm[0,1]:4d}")
print(f"       Malignant  {cm[1,0]:4d}    {cm[1,1]:4d}")

# Calculate metrics
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\nDetailed Metrics:")
print(f"  Sensitivity (Recall):    {sensitivity:.4f}")
print(f"  Specificity:             {specificity:.4f}")
print(f"  Positive Predictive Value: {ppv:.4f}")
print(f"  Negative Predictive Value: {npv:.4f}")

## Step 12: Visualize ROC and PR Curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(breast_labels, breast_preds)
axes[0].plot(fpr, tpr, color='blue', linewidth=2, label=f'AUROC = {breast_auroc:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve (Breast-Level)', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(breast_labels, breast_preds)
baseline = (breast_labels == 1).sum() / len(breast_labels)
axes[1].plot(recall, precision, color='green', linewidth=2, label=f'PR-AUC = {breast_pr_auc:.4f}')
axes[1].axhline(y=baseline, color='k', linestyle='--', linewidth=1, label=f'Baseline = {baseline:.4f}')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve (Breast-Level)', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ ROC and PR curves plotted")

## Step 13: Visualize Sample Predictions

In [ ]:
# Sample diverse predictions
np.random.seed(42)

# Get some true positives, true negatives, false positives, false negatives
tp_mask = (breast_preds_binary == 1) & (breast_labels == 1)
tn_mask = (breast_preds_binary == 0) & (breast_labels == 0)
fp_mask = (breast_preds_binary == 1) & (breast_labels == 0)
fn_mask = (breast_preds_binary == 0) & (breast_labels == 1)

# Get breast IDs for each category
breast_ids = list(breast_preds_dict.keys())

def get_samples(mask, n=2):
    indices = np.where(mask)[0]
    if len(indices) > 0:
        selected = np.random.choice(indices, min(n, len(indices)), replace=False)
        return [breast_ids[i] for i in selected]
    return []

samples = {
    'True Positive': get_samples(tp_mask, 3),
    'True Negative': get_samples(tn_mask, 3),
    'False Positive': get_samples(fp_mask, 2),
    'False Negative': get_samples(fn_mask, 2)
}

# Visualize
for category, breast_ids_sample in samples.items():
    if len(breast_ids_sample) == 0:
        continue
    
    print(f"\n{'='*70}")
    print(f"{category.upper()}")
    print(f"{'='*70}")
    
    for breast_id in breast_ids_sample:
        # Get breast info
        breast_pred = breast_preds[breast_ids.index(breast_id)]
        breast_label = breast_labels[breast_ids.index(breast_id)]
        
        # Get images for this breast
        breast_images = test_df[test_df['breast_id'] == breast_id]
        
        print(f"\nBreast ID: {breast_id}")
        print(f"  True Label:  {'MALIGNANT' if breast_label == 1 else 'BENIGN'}")
        print(f"  Prediction:  {breast_pred:.4f} ({'MALIGNANT' if breast_pred >= 0.5 else 'BENIGN'})")
        print(f"  # Images:    {len(breast_images)}")
        
        # Show image-level predictions
        for idx, row in breast_images.iterrows():
            img_id = row['image_id']
            img_pred = image_predictions.get(img_id, 0.0)
            view = row.get('view_position', row.get('view', 'Unknown'))
            print(f"    [{view}] {img_id}: {img_pred:.4f}")

print(f"\n{'='*70}")

## Step 14: Save Predictions

In [ ]:
# Create results dataframe
results_df = test_df.copy()
results_df['prediction'] = results_df['image_id'].map(image_predictions)

# Add breast-level predictions
breast_pred_map = {}
for i, breast_id in enumerate(breast_ids):
    breast_pred_map[breast_id] = breast_preds[i]

results_df['breast_prediction'] = results_df['breast_id'].map(breast_pred_map)

# Save to CSV
output_dir = Path(CHECKPOINT_PATH).parent
results_path = output_dir / 'test_predictions.csv'
results_df.to_csv(results_path, index=False)

print(f"✅ Predictions saved to: {results_path}")
print(f"   Columns: {list(results_df.columns)}")
print(f"\nFirst few rows:")
results_df.head()

## Summary

✅ **Inference complete!**

**Image-Level Metrics:**
- Predictions on individual mammogram views
- Useful for understanding model's raw performance

**Breast-Level Metrics (Noisy-OR):**
- Aggregates multiple views per breast
- More clinically relevant evaluation
- Used for final diagnostic decision

**Outputs:**
- Test predictions: `{output_dir}/test_predictions.csv`
- ROC and PR curves visualized
- Confusion matrix and classification report

**Next steps:**
- Adjust classification threshold based on clinical needs
- Test on external dataset (e.g., INbreast)
- Analyze errors for model improvement